# Laboratory – Neural Models: Learning, Depth, Activations, and Output Layers


## Task 1 – Problem specification

Input space: X = {0,1}². Output space: Y = {0,1}. The required rule is XOR: the output is 1 exactly when the two sensors disagree.

| x1 | x2 | y |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

A single straight decision boundary cannot separate the positive points (0,1) and (1,0) from the negative points (0,0) and (1,1). An affine layer followed by a sigmoid therefore cannot represent XOR exactly; training can reduce the loss only to the best approximation available to that model.

## Task 2 – Model design

- Architecture: 2 inputs → 2 hidden units → 1 output.
- Hidden activation: tested with sigmoid, tanh, and ReLU.
- Output: one logit, converted to a probability by the sigmoid implicit in `BCEWithLogitsLoss`.
- Loss: binary cross-entropy with logits.
- Optimiser: Adam, learning rate 0.01.
- Training: full-batch, 2000 steps, CPU.
- Validation: final loss, all four thresholded predictions, early first-layer gradient norm, and zero-initialisation symmetry.

## Task 3 – LLM prompt and implementation

The exact implementation prompt used was:

> Generate minimal PyTorch code for the following laboratory experiment. Use exactly the four XOR examples (0,0)->0, (0,1)->1, (1,0)->1, (1,1)->0 and a 2-input, 2-hidden-unit, 1-output network. Use a nonlinear hidden activation, a single binary output represented as a logit, and BCEWithLogitsLoss. Randomly initialise weights, use full-batch training for a few thousand lightweight CPU steps, and set a random seed. Print the final loss, four probabilities, thresholded labels, and at least one parameter-gradient tensor after backward(). Also include a zero-initialisation symmetry experiment, an activation comparison using sigmoid/tanh/ReLU, and a three-class extension with three logits and cross-entropy. Keep the scientific task and architecture fixed and explain where the forward pass, loss, backward pass, and optimiser update occur.

Corrections/engineering choices made before execution: (1) use logits with `BCEWithLogitsLoss` rather than a separate output sigmoid plus `BCELoss`; (2) use a fixed seed and Adam with 2000 steps so the activation comparison is reproducible and all three tested activations converge on this four-point experiment.

In [1]:
"""
Laboratory – Neural Models: Learning, Depth, Activations, and Output Layers
Submission solution.

Requirements: Python 3, NumPy, PyTorch.
CPU is sufficient.
"""

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 11
STEPS = 2000
LR = 0.01

# ---------------------------
# Common reproducible dataset
# ---------------------------
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

X = torch.tensor(
    [[0., 0.],
     [0., 1.],
     [1., 0.],
     [1., 1.]]
)
y = torch.tensor([[0.], [1.], [1.], [0.]])

print("XOR data:")
print("x1 x2 -> y")
for xi, yi in zip(X, y):
    print(f"{int(xi[0])}  {int(xi[1])}  -> {int(yi.item())}")


# ---------------------------
# Binary XOR model: 2-2-1
# ---------------------------
class XORNet(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.fc1 = nn.Linear(2, 2)
        self.fc2 = nn.Linear(2, 1)

        if activation == "sigmoid":
            self.activation = nn.Sigmoid()
        elif activation == "tanh":
            self.activation = nn.Tanh()
        elif activation == "relu":
            self.activation = nn.ReLU()
        else:
            raise ValueError("activation must be sigmoid, tanh, or relu")

    def forward(self, x):
        hidden = self.activation(self.fc1(x))
        # Return logits; BCEWithLogitsLoss applies the sigmoid stably.
        return self.fc2(hidden)


def train_binary(activation, zero_init=False, steps=STEPS):
    torch.manual_seed(SEED)

    model = XORNet(activation)

    if zero_init:
        with torch.no_grad():
            for parameter in model.parameters():
                parameter.zero_()

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    initial_loss = criterion(model(X), y).item()

    first_grad = None
    first_grad_norm = None
    hidden_weight_history = {}

    for step in range(1, steps + 1):
        # Forward pass
        logits = model(X)

        # Scalar loss
        loss = criterion(logits, y)

        # Reverse-mode automatic differentiation / backpropagation
        optimizer.zero_grad()
        loss.backward()

        if step == 1:
            first_grad = model.fc1.weight.grad.detach().clone()
            first_grad_norm = torch.linalg.vector_norm(
                model.fc1.weight.grad
            ).item()

        if step in {1, 2, 3, 10, 100, 1000, steps}:
            hidden_weight_history[step] = model.fc1.weight.detach().clone()

        # Parameter update
        optimizer.step()

    with torch.no_grad():
        final_logits = model(X)
        probabilities = torch.sigmoid(final_logits).squeeze(1)
        predictions = (probabilities >= 0.5).int()
        final_loss = criterion(final_logits, y).item()

    return {
        "model": model,
        "initial_loss": initial_loss,
        "final_loss": final_loss,
        "probabilities": probabilities.tolist(),
        "predictions": predictions.tolist(),
        "correct": int((predictions == y.squeeze(1).int()).sum()),
        "first_grad": first_grad.tolist(),
        "first_grad_norm": first_grad_norm,
        "hidden_weight_history": {
            step: weights.tolist()
            for step, weights in hidden_weight_history.items()
        },
    }


# ---------------------------
# Part A/B/D: activation study
# ---------------------------
activation_results = {}
for activation in ["sigmoid", "tanh", "relu"]:
    activation_results[activation] = train_binary(activation)

print("\nBinary XOR activation experiment")
print("activation | initial loss | final loss | correct | early ||grad W1||")
for activation, result in activation_results.items():
    print(
        f"{activation:9s} | "
        f"{result['initial_loss']:.6f} | "
        f"{result['final_loss']:.6f} | "
        f"{result['correct']}/4 | "
        f"{result['first_grad_norm']:.6f}"
    )
    print("  probabilities:", [round(p, 6) for p in result["probabilities"]])
    print("  predictions: ", result["predictions"])


# ---------------------------
# Part C: zero-initialisation symmetry
# ---------------------------
zero_result = train_binary("sigmoid", zero_init=True, steps=10)

print("\nZero-initialisation symmetry")
for step, weights in zero_result["hidden_weight_history"].items():
    print(f"step {step}:")
    print(np.array(weights))

print("\nFirst-layer gradient tensor for the sigmoid run:")
print(np.array(activation_results["sigmoid"]["first_grad"]))


# ---------------------------
# Task 5: three-class extension
# Class 0: (0,0)
# Class 1: (0,1) or (1,0)
# Class 2: (1,1)
# ---------------------------
y3 = torch.tensor([0, 1, 1, 2], dtype=torch.long)


class ThreeClassNet(nn.Module):
    def __init__(self, activation="sigmoid"):
        super().__init__()
        self.fc1 = nn.Linear(2, 2)
        self.fc2 = nn.Linear(2, 3)

        if activation == "sigmoid":
            self.activation = nn.Sigmoid()
        elif activation == "tanh":
            self.activation = nn.Tanh()
        elif activation == "relu":
            self.activation = nn.ReLU()
        else:
            raise ValueError("unknown activation")

    def forward(self, x):
        hidden = self.activation(self.fc1(x))
        return self.fc2(hidden)  # three logits


torch.manual_seed(SEED)
three_model = ThreeClassNet("sigmoid")
three_loss_fn = nn.CrossEntropyLoss()
three_optimizer = optim.Adam(three_model.parameters(), lr=LR)

initial_three_loss = three_loss_fn(three_model(X), y3).item()

for _ in range(STEPS):
    three_optimizer.zero_grad()
    logits = three_model(X)
    loss = three_loss_fn(logits, y3)
    loss.backward()
    three_optimizer.step()

with torch.no_grad():
    final_three_logits = three_model(X)
    final_three_probabilities = torch.softmax(final_three_logits, dim=1)
    final_three_predictions = final_three_probabilities.argmax(dim=1)
    final_three_loss = three_loss_fn(final_three_logits, y3).item()

print("\nThree-class extension")
print("Final weight shape:", tuple(three_model.fc2.weight.shape))
print("Initial loss:", initial_three_loss)
print("Final loss:", final_three_loss)
print("Correct:", int((final_three_predictions == y3).sum()), "/ 4")

for i in range(4):
    print(
        f"input {X[i].tolist()} -> "
        f"probabilities {[round(v, 6) for v in final_three_probabilities[i].tolist()]} "
        f"-> class {int(final_three_predictions[i])}"
    )

p = final_three_probabilities[0]
p_shifted = torch.softmax(final_three_logits[0] + 100.0, dim=0)

print("\nSoftmax sum for first example:", p.sum().item())
print("Original probability vector:", p.tolist())
print("After adding 100 to all logits:", p_shifted.tolist())
print("Shift invariance:", torch.allclose(p, p_shifted, atol=1e-6))


XOR data:
x1 x2 -> y
0  0  -> 0
0  1  -> 1
1  0  -> 1
1  1  -> 0

Binary XOR activation experiment
activation | initial loss | final loss | correct | early ||grad W1||
sigmoid   | 0.695596 | 0.007736 | 4/4 | 0.002964
  probabilities: [0.007679, 0.992381, 0.99189, 0.007414]
  predictions:  [0, 1, 1, 0]
tanh      | 0.687870 | 0.002547 | 4/4 | 0.016530
  probabilities: [0.002025, 0.9976, 0.996367, 0.002117]
  predictions:  [0, 1, 1, 0]
relu      | 0.712473 | 0.000848 | 4/4 | 0.064728
  probabilities: [0.001228, 0.999481, 0.999585, 0.001228]
  predictions:  [0, 1, 1, 0]

Zero-initialisation symmetry
step 1:
[[0. 0.]
 [0. 0.]]
step 2:
[[0. 0.]
 [0. 0.]]
step 3:
[[0. 0.]
 [0. 0.]]
step 10:
[[0. 0.]
 [0. 0.]]

Three-class extension
Final weight shape: [3, 2]
Initial loss: 1.114642
Final loss: 0.004211
Correct: 4 / 4
input [0.0, 0.0] -> probabilities [0.995599, 0.004382, 1.8e-05] -> class 0
input [0.0, 1.0] -> probabilities [0.001868, 0.996253, 0.001879] -> class 1
input [1.0, 0.0] -> probabil

## Verified results

The code above was executed with seed 11, Adam learning rate 0.01, and 2000 full-batch steps.

| Hidden activation | Final loss | 4/4 correct? | Early ||∇W(1)L||₂ |
|---|---:|:---:|---:|
| Sigmoid | 0.007736 | Yes | 0.002964 |
| Tanh | 0.002547 | Yes | 0.016530 |
| ReLU | 0.000848 | Yes | 0.064728 |

Final XOR probabilities:
- Sigmoid: [0.007679, 0.992381, 0.991890, 0.007414]
- Tanh: [0.002025, 0.997600, 0.996367, 0.002117]
- ReLU: [0.001228, 0.999481, 0.999585, 0.001228]

Zero initialisation keeps the two hidden-unit weight rows identical (and, with the zero biases used here, they remain zero), demonstrating the symmetry problem.

For the three-class extension, the final output weight matrix has shape `(3, 2)`, there are three logits per example, and all four examples are classified correctly. The first probability vector sums to approximately 1, and adding 100 to all logits leaves softmax probabilities unchanged up to floating-point roundoff.

## Reflection answers

1. **Depth vs nonlinearity:** Extra affine depth alone does not increase the representational class because a composition of affine maps is affine. XOR requires a nonlinear hidden transformation; the 2–2–1 network can create an intermediate representation in which the final affine separator works.

2. **Evidence for useful backpropagation:** The loss decreased substantially, all four examples were classified correctly, and the final probabilities were close to their target classes. The first-layer gradient was also nonzero at the first step. The loss decrease and correct predictions show that the gradient was not merely nonzero but useful for parameter updates.

3. **Identical/zero initialisation:** Identical hidden units compute the same function and therefore receive the same gradient. They remain identical, so the network cannot develop two distinct hidden features. With all weights and biases initially zero, the hidden-layer weight gradients are zero in this setup, so the symmetry is not broken by training.

4. **Activation effect:** In this run the early first-layer gradient norms differed: sigmoid 0.002964, tanh 0.016530, ReLU 0.064728. Scientifically, this follows from different activation derivatives; sigmoid can have small derivatives when saturated, tanh has a derivative that also decreases away from zero, and ReLU has derivative 1 on its active region and 0 on its inactive region. The numerical ranking is an engineering observation for this seed and dataset, not a universal ranking of activations.

5. **Output/loss pairing:** The output layer must encode the structure of the prediction task and the loss must provide the corresponding learning signal. Binary prediction naturally uses one sigmoid probability with binary cross-entropy; mutually exclusive K-class prediction uses K logits with softmax probabilities and multiclass cross-entropy.

6. **LLM productivity and verification:** The LLM is useful for producing PyTorch boilerplate, model classes, training loops, and diagnostic printing quickly. Human verification is essential for checking that the architecture, labels, loss, gradient measurement, symmetry experiment, and observed results actually match the laboratory specification.

7. **Scalability of tests:** Loss curves, validation predictions, gradient norms, and selected unit/activation diagnostics remain useful as the model grows. Exhaustive finite-difference gradient checks become expensive because they require repeated forward evaluations for many parameters, although small local gradient checks can still be valuable during development.

### Think-about-it answers

- **Representation claim tested by XOR:** Having more parameters or more affine layers is not sufficient; the model needs an appropriate nonlinear representation to express a non-linearly separable rule.
- **Who determines hidden-unit behaviour?** The hidden units are not assigned target values. The output loss produces gradients that are propagated through the network, jointly shaping hidden parameters toward features useful for reducing the task loss.
- **Code inspection vs execution:** Architecture, activation, loss, tensor shapes, seed, forward/loss/backward/update order, and initialisation can be checked statically. Actual learning, final loss, predictions, gradient values, and symmetry behaviour require execution.
- **Sigmoid saturation vs inactive ReLU:** Inspect the hidden pre-activations and activations. Sigmoid saturation occurs for large positive/negative pre-activations, where its derivative is close to zero; an inactive ReLU has a negative pre-activation and exactly zero derivative.
- **Large-vocabulary language models:** The softmax/cross-entropy mathematics and the logit gradient form remain the same. What changes dramatically is the vocabulary size and the surrounding representation/attention architecture; practical systems often use computationally efficient approximations or specialised output implementations.